# Day 15 – LangChain Basics

Covers: prompt templates, LLM chains, sequential chains, conversation memory, and structured output parsing.

In [25]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(dotenv_path="../.env") 

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
print("LLM ready:", llm.model_name)

LLM ready: gpt-4o-mini


In [26]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate(
    input_variables=["topic", "audience"],
    template="Explain {topic} to a {audience} in 2 simple sentences."
)

filled = template.format(topic="neural networks", audience="10-year-old")
print("Filled prompt:")
print(filled)

Filled prompt:
Explain neural networks to a 10-year-old in 2 simple sentences.


In [27]:
from langchain_core.prompts import ChatPromptTemplate

# ChatPromptTemplate supports system + user messages
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant who explains things simply."),
    ("human",  "Explain {topic} to a {audience} in 2 simple sentences.")
])

messages = chat_template.format_messages(topic="machine learning", audience="manager")
for m in messages:
    print(f"{m.type.upper()}: {m.content}")

SYSTEM: You are a helpful assistant who explains things simply.
HUMAN: Explain machine learning to a manager in 2 simple sentences.


In [28]:
from langchain_core.output_parsers import StrOutputParser

# Build the chain: prompt → llm → parse to string
prompt = ChatPromptTemplate.from_messages([
    ("human", "Give me a one-sentence definition of {concept}.")
])

chain = prompt | llm | StrOutputParser()

# Run the chain
result = chain.invoke({"concept": "overfitting"})
print(result)

Overfitting is a modeling error that occurs when a machine learning model learns the training data too well, capturing noise and outliers instead of the underlying pattern, leading to poor generalization on unseen data.


In [29]:
# Batch mode – run the same chain on multiple inputs at once
concepts = [
    {"concept": "gradient descent"},
    {"concept": "tokenization"},
    {"concept": "attention mechanism"},
]

results = chain.batch(concepts)
for concept, result in zip(concepts, results):
    print(f"{concept['concept']:25s} → {result}")

gradient descent          → Gradient descent is an optimization algorithm used to minimize a function by iteratively adjusting its parameters in the direction of the steepest descent, as indicated by the negative gradient.
tokenization              → Tokenization is the process of converting sensitive data into non-sensitive tokens that can be used in place of the original data for security and privacy purposes.
attention mechanism       → An attention mechanism is a computational technique in neural networks that allows the model to focus on specific parts of the input data when making predictions, thereby improving performance on tasks like natural language processing and image recognition.


---
## 4. Sequential Chain (Multi-Step Processing)

Chain multiple LLM calls where **output of step 1 becomes input of step 2**.

Example pipeline: `topic → draft email → improve tone`

In [30]:
parser = StrOutputParser()

# Step 1: Write a rough draft
draft_prompt = ChatPromptTemplate.from_messages([
    ("human", "Write a short email (3 sentences) about: {topic}")
])
draft_chain = draft_prompt | llm | parser

# Step 2: Polish the draft
polish_prompt = ChatPromptTemplate.from_messages([
    ("human", "Improve the tone of this email to sound more professional:\n\n{draft}")
])
polish_chain = polish_prompt | llm | parser

# Connect: output of draft_chain feeds into polish_chain as {draft}
sequential_chain = draft_chain | (lambda draft: {"draft": draft}) | polish_chain

final = sequential_chain.invoke({"topic": "project deadline extension request"})
print("=== Final polished email ===")
print(final)

=== Final polished email ===
Subject: Request for Extension on Project Deadline

Dear [Recipient's Name],  

I hope this message finds you well. I am writing to formally request an extension on the deadline for our current project due to unforeseen circumstances that have arisen. This additional time will allow us to maintain our commitment to delivering the highest quality work.  

I appreciate your understanding and consideration of this request. I look forward to your response.  

Sincerely,  
[Your Name]  


In [31]:
# Three-step chain: topic → outline → full article → summary
outline_prompt = ChatPromptTemplate.from_messages([
    ("human", "Create a 3-point outline for a blog post about: {topic}")
])

expand_prompt = ChatPromptTemplate.from_messages([
    ("human", "Write a short blog introduction (2 sentences) based on this outline:\n{outline}")
])

summarize_prompt = ChatPromptTemplate.from_messages([
    ("human", "Summarize this in one sentence:\n{intro}")
])

chain_3step = (
    outline_prompt | llm | parser
    | (lambda x: {"outline": x})
    | expand_prompt | llm | parser
    | (lambda x: {"intro": x})
    | summarize_prompt | llm | parser
)

result = chain_3step.invoke({"topic": "benefits of daily exercise"})
print("3-step chain result:")
print(result)

3-step chain result:
Daily exercise provides numerous benefits that enhance physical health, mental well-being, and social connections, making it a transformative addition to any routine.


---
## 5. Conversation Memory

`ConversationBufferMemory` stores the full chat history and automatically
injects it into each prompt, so the model remembers previous messages.

In [32]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

memory = ConversationBufferMemory()

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False  # set True to see full prompt with history
)

# Turn 1
r1 = conversation.predict(input="Hi, my name is Pradhnyesh and I'm learning LangChain.")
print("Bot:", r1)

# Turn 2 – model should remember the name
r2 = conversation.predict(input="What's my name and what am I learning?")
print("Bot:", r2)

ModuleNotFoundError: No module named 'langchain.memory'

In [ ]:
# Inspect what's stored in memory
print(memory.buffer)

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

# Window memory – only remembers last k exchanges (saves tokens)
window_memory = ConversationBufferWindowMemory(k=2)  # last 2 turns only

window_conv = ConversationChain(llm=llm, memory=window_memory, verbose=False)

window_conv.predict(input="My favourite language is Python.")
window_conv.predict(input="I also like JavaScript.")
window_conv.predict(input="And I enjoy SQL too.")  # this pushes Python out of window

r = window_conv.predict(input="List all the languages I mentioned.")
print("Bot:", r)
print("\nNote: Python may be forgotten — only last 2 turns are kept.")

---
## 6. Structured Output Parsing

Output parsers convert raw LLM text into Python objects (dicts, lists) automatically.

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

list_parser = CommaSeparatedListOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("human", "List 5 {category}. {format_instructions}")
])

chain = prompt | llm | list_parser

result = chain.invoke({
    "category": "popular Python libraries for data science",
    "format_instructions": list_parser.get_format_instructions()
})

print(type(result))   # <class 'list'>
print(result)
for i, item in enumerate(result, 1):
    print(f"  {i}. {item.strip()}")

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("human",
     "Extract information from this text and return as JSON with keys: "
     "name, age, city, job.\n\nText: {text}\n\nReturn only valid JSON.")
])

chain = prompt | llm | json_parser

result = chain.invoke({
    "text": "Ravi is a 28-year-old software engineer living in Pune."
})

print(type(result))   # <class 'dict'>
print(result)
print("Name:", result["name"])
print("City:", result["city"])

In [ ]:
# Parse a more complex JSON structure
prompt2 = ChatPromptTemplate.from_messages([
    ("human",
     "Analyze this product review and return JSON with keys: "
     "sentiment (positive/negative/neutral), score (1-5), pros (list), cons (list).\n\n"
     "Review: {review}\n\nReturn only valid JSON.")
])

analysis_chain = prompt2 | llm | json_parser

result2 = analysis_chain.invoke({
    "review": "Great camera and battery life, but the phone gets warm during gaming and the price is high."
})

print(result2)
print("\nSentiment:", result2["sentiment"])
print("Pros:", result2["pros"])
print("Cons:", result2["cons"])